# Notebook 01 — The Qubit and Superposition

**Prerequisites:** Go programming basics and a working gonb installation (see setup below).

Welcome to the first notebook in our 16-part quantum computing curriculum built on the **Goqu** SDK. By the end of this notebook you will be able to:

1. Describe what a qubit is mathematically (a 2D complex vector).
2. Build and visualize quantum circuits in Go.
3. Explain superposition, probability amplitudes, and the Born rule.
4. Demonstrate, with code, that superposition is **not** a coin flip.

---

### Prerequisites

You need the **gonb** Go kernel for Jupyter. Install it with:

```bash
go install github.com/janpfeifer/gonb@latest
gonb --install
```

Then open this notebook in JupyterLab or VS Code and select the **Go (gonb)** kernel.

In [1]:
import (
	"fmt"
	"math"
	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/sim/statevector"
	"github.com/splch/goqu/viz"
)

## What is a qubit?

A classical bit is either 0 or 1. A **qubit** (quantum bit) is a two-dimensional complex vector:

$$|\psi\rangle = \alpha\,|0\rangle + \beta\,|1\rangle$$

where $\alpha$ and $\beta$ are **probability amplitudes** — complex numbers satisfying $|\alpha|^2 + |\beta|^2 = 1$.

The **computational basis** consists of two orthogonal states:

| State | Vector | Meaning |
|-------|--------|---------|
| $\|0\rangle$ | $\begin{pmatrix}1\\0\end{pmatrix}$ | "off", ground state |
| $\|1\rangle$ | $\begin{pmatrix}0\\1\end{pmatrix}$ | "on", excited state |

When we **measure** a qubit, we get outcome 0 with probability $|\alpha|^2$ and outcome 1 with probability $|\beta|^2$. This is the **Born rule**.

The Hadamard gate **H** transforms $|0\rangle$ into the **equal superposition** state:

$$H|0\rangle = \frac{1}{\sqrt{2}}|0\rangle + \frac{1}{\sqrt{2}}|1\rangle \;=\; |+\rangle$$

In this state, each outcome has probability $(1/\sqrt{2})^2 = 1/2$.

## Your first circuit

Let's build a single-qubit circuit that applies the Hadamard gate, then print its text diagram.

The Goqu builder uses a **fluent API**:
- `builder.New(name, nQubits)` creates a new circuit builder.
- `.H(0)` applies Hadamard to qubit 0.
- `.Build()` finalizes the circuit.
- `draw.String(circuit)` renders an ASCII diagram.

In [2]:
%%
c, err := builder.New("hadamard", 1).H(0).Build()
if err != nil {
	fmt.Println("Error:", err)
} else {
	gonbui.DisplayHTML(draw.SVG(c))
}

q0 
 
 H

## Inspecting the statevector

A **statevector simulator** tracks the exact amplitudes of every basis state. There is no randomness until we measure.

After applying H to $|0\rangle$, we expect the statevector to be:

$$\left[\frac{1}{\sqrt{2}},\; \frac{1}{\sqrt{2}}\right] \approx [0.707,\; 0.707]$$

Let's verify with `statevector.New(1)` (one qubit, initialized to $|0\rangle$).

In [3]:
%%
sim := statevector.New(1)
c, _ := builder.New("h", 1).H(0).Build()
sim.Evolve(c)
sv := sim.StateVector()
fmt.Printf("Statevector: %v\n", sv)
fmt.Printf("|0> amplitude: %.6f   probability: %.4f\n", real(sv[0]), real(sv[0])*real(sv[0]))
fmt.Printf("|1> amplitude: %.6f   probability: %.4f\n", real(sv[1]), real(sv[1])*real(sv[1]))

Statevector: [(0.7071067811865476+0i) (0.7071067811865476+0i)]
|0> amplitude: 0.707107   probability: 0.5000
|1> amplitude: 0.707107   probability: 0.5000


## Visualizing on the Bloch sphere

Every single-qubit pure state can be represented as a point on the **Bloch sphere**:

- $|0\rangle$ is the **north pole**.
- $|1\rangle$ is the **south pole**.
- $|+\rangle$ sits on the **equator** (pointing along the +X axis).

The `viz.Bloch` function takes a 2-element statevector and returns an SVG rendering of the sphere.

In [4]:
%%
sim := statevector.New(1)
c, _ := builder.New("bloch-plus", 1).H(0).Build()
sim.Evolve(c)
svg := viz.Bloch(sim.StateVector(), viz.WithTitle("|+> state on the Bloch sphere"))
gonbui.DisplayHTML(svg)

|+> state on the Bloch sphere 
<path d="M438.6,230.6 L445.0,225.9 L450.4,220.9 L454.5,215.8 L457.6,210.6 L459.4,205.3 L460.0,200.0 L459.4,194.7 L457.6,189.4 L454.5,184.2 L450.4,179.1 L445.0,174.1 L438.6,169.4 L431.1,164.9 L422.6,160.6 L413.1,156.7 L402.8,153.1 L391.8,149.8 L380.0,147.0 L367.6,144.5 L354.7,142.5 L341.4,140.9 L327.8,139.7 L313.9,139.0 L300.0,138.8 L286.1,139.0 L272.2,139.7 L258.6,140.9 L245.3,142.5 L232.4,144.5 L220.0,147.0 L208.2,149.8 L197.2,153.1 L186.9,156.7 L177.4,160.6 L168.9,164.9 L161.4,169.4 L155.0,174.1 L149.6,179.1 L145.5,184.2 L142.4,189.4 L140.6,194.7 L140.0,200.0 L140.6,205.3 L142.4,210.6 L145.5,215.8 L149.6,220.9 L155.0,225.9 L161.4,230.6 L168.9,235.1 L177.4,239.4 L186.9,243.3 L197.2,246.9 L208.2,250.2 L220.0,253.0 L232.4,255.5 L245.3,257.5 L258.6,259.1 L272.2,260.3 L286.1,261.0 L300.0,261.2 L313.9,261.0 L327.8,260.3 L341.4,259.1 L354.7,257.5 L367.6,255.5 L380.0,253.0 L391.8,250.2 L402.8,246.9 L413.1,243.3 L422.6,239.4 L431.1,235.1 L438.6,230.6 Z" fill="none" stroke="#AAAAAA" stroke-width="0.5" opacity="0.4"/>
<path d="M438.6,230.6 L438.0,217.6 L436.5,204.5 L433.8,191.3 L430.2,178.2 L425.6,165.3 L420.0,152.6 L413.5,140.3 L406.1,128.4 L398.0,117.1 L389.1,106.4 L379.5,96.5 L369.3,87.3 L358.6,79.0 L347.4,71.6 L335.9,65.1 L324.1,59.7 L312.1,55.4 L300.0,52.2 L287.9,50.1 L275.9,49.1 L264.1,49.3 L252.6,50.6 L241.4,53.1 L230.7,56.7 L220.5,61.4 L210.9,67.1 L202.0,73.8 L193.9,81.5 L186.5,90.1 L180.0,99.6 L174.4,109.8 L169.8,120.7 L166.2,132.2 L163.5,144.2 L162.0,156.6 L161.4,169.4 L162.0,182.4 L163.5,195.5 L166.2,208.7 L169.8,221.8 L174.4,234.7 L180.0,247.4 L186.5,259.7 L193.9,271.6 L202.0,282.9 L210.9,293.6 L220.5,303.5 L230.7,312.7 L241.4,321.0 L252.6,328.4 L264.1,334.9 L275.9,340.3 L287.9,344.6 L300.0,347.8 L312.1,349.9 L324.1,350.9 L335.9,350.7 L347.4,349.4 L358.6,346.9 L369.3,343.3 L379.5,338.6 L389.1,332.9 L398.0,326.2 L406.1,318.5 L413.5,309.9 L420.0,300.4 L425.6,290.2 L430.2,279.3 L433.8,267.8 L436.5,255.8 L438.0,243.4 L438.6,230.6 Z" fill="none" stroke="#AAAAAA" stroke-width="0.5" opacity="0.4"/>
<path d="M380.0,147.0 L379.7,134.3 L378.8,122.1 L377.3,110.5 L375.2,99.6 L372.5,89.5 L369.3,80.2 L365.5,71.8 L361.3,64.4 L356.6,58.0 L351.4,52.7 L345.9,48.5 L340.0,45.5 L333.8,43.6 L327.4,43.0 L320.7,43.5 L313.9,45.2 L307.0,48.1 L300.0,52.2 L293.0,57.4 L286.1,63.6 L279.3,70.9 L272.6,79.2 L266.2,88.4 L260.0,98.5 L254.1,109.3 L248.6,120.8 L243.4,133.0 L238.7,145.6 L234.5,158.7 L230.7,172.0 L227.5,185.6 L224.8,199.3 L222.7,213.0 L221.2,226.6 L220.3,239.9 L220.0,253.0 L220.3,265.7 L221.2,277.9 L222.7,289.5 L224.8,300.4 L227.5,310.5 L230.7,319.8 L234.5,328.2 L238.7,335.6 L243.4,342.0 L248.6,347.3 L254.1,351.5 L260.0,354.5 L266.2,356.4 L272.6,357.0 L279.3,356.5 L286.1,354.8 L293.0,351.9 L300.0,347.8 L307.0,342.6 L313.9,336.4 L320.7,329.1 L327.4,320.8 L333.8,311.6 L340.0,301.5 L345.9,290.7 L351.4,279.2 L356.6,267.0 L361.3,254.4 L365.5,241.3 L369.3,228.0 L372.5,214.4 L375.2,200.7 L377.3,187.0 L378.8,173.4 L379.7,160.1 L380.0,147.0 Z" fill="none" stroke="#AAAAAA" stroke-width="0.5" opacity="0.4"/>
 
 
 
 
 |+⟩ 
 |-⟩ 
 |+i⟩ 
 |-i⟩ 
 |0⟩ 
 |1⟩

## Measurement and the Born rule

When we add `.MeasureAll()` to the circuit and run it for many **shots**, each shot collapses the superposition into a definite outcome. The fraction of shots yielding each outcome converges to the Born-rule probabilities.

For $|+\rangle$, we expect roughly 50% "0" and 50% "1".

In [5]:
%%
c, _ := builder.New("measure-plus", 1).H(0).MeasureAll().Build()
sim := statevector.New(1)
counts, _ := sim.Run(c, 1000)
fmt.Println("Counts:", counts)
gonbui.DisplayHTML(viz.Histogram(counts, viz.WithTitle("H|0> — 1000 shots")))

Counts: map[0:487 1:513]


H|0> — 1000 shots 
 
 
 
 0 
 
 100 
 
 200 
 
 300 
 
 400 
 
 500 
 
 600 
 
 0 
 
 1

## Predict, then verify

Before running the next cell, **pause and predict**: what will the measurement outcomes be for a circuit that applies only an **X gate** (the quantum NOT gate) to $|0\rangle$?

The X gate flips a qubit:

$$X|0\rangle = |1\rangle, \qquad X|1\rangle = |0\rangle$$

So we should **always** get outcome "1" — 100% of the time.

In [6]:
%%
c, _ := builder.New("x-gate", 1).X(0).MeasureAll().Build()
gonbui.DisplayHTML(draw.SVG(c))
sim := statevector.New(1)
counts, _ := sim.Run(c, 1000)
fmt.Println("Counts:", counts)

Counts: map[1:1000]


q0 
 
 X 
 M

## Superposition is NOT a coin flip

A common misconception is that a qubit in superposition is "just a coin flip — we don't know the answer yet, but it's already decided." This is wrong. A qubit in superposition carries **phase information** that can produce **interference**, something no classical coin can do.

### The proof: H followed by H

If superposition were merely classical ignorance (a coin flip), then:
- H would create a 50/50 coin.
- A second H would flip the coin again — still 50/50.

But quantum mechanics says:

$$H \cdot H |0\rangle = |0\rangle$$

The two Hadamard gates produce **constructive interference** for $|0\rangle$ and **destructive interference** for $|1\rangle$, perfectly canceling the $|1\rangle$ amplitude. The result is deterministically $|0\rangle$ — 100% of the time.

Mathematically:

$$H|+\rangle = H\left(\frac{|0\rangle + |1\rangle}{\sqrt{2}}\right) = \frac{H|0\rangle + H|1\rangle}{\sqrt{2}} = \frac{(|0\rangle + |1\rangle) + (|0\rangle - |1\rangle)}{2} = |0\rangle$$

Let's verify this and contrast it with what a "coin flip" model would predict.

## Self-Check Questions

Before attempting the exercises, make sure you can answer these:

1. True or false: A qubit in superposition is in state |0⟩ AND state |1⟩ at the same time. Why or why not?
2. Why does H·H|0⟩ = |0⟩, but a coin flipped twice still gives a 50/50 outcome?
3. If a qubit is in state |ψ⟩ = √(0.3)|0⟩ + √(0.7)|1⟩, what is the probability of measuring |0⟩?

In [7]:
%%
// Quantum: H -> H should give |0> with 100% probability (interference!)
cHH, _ := builder.New("H-H interference", 1).H(0).H(0).MeasureAll().Build()
fmt.Println("Circuit: H -> H")
gonbui.DisplayHTML(draw.SVG(cHH))

sim := statevector.New(1)

// Check the statevector before measurement
cHHnoMeasure, _ := builder.New("H-H sv", 1).H(0).H(0).Build()
sim.Evolve(cHHnoMeasure)
fmt.Printf("Statevector after H->H: %v\n\n", sim.StateVector())

// Run with measurement
counts, _ := sim.Run(cHH, 1000)
fmt.Println("Quantum result (H->H, 1000 shots):")
fmt.Println(counts)
fmt.Println()
fmt.Println("If superposition were a coin flip, we'd expect ~500 '0' and ~500 '1'.")
fmt.Println("Instead we get 1000 '0' and 0 '1' — proof of quantum interference!")

Circuit: H -> H
Statevector after H->H: [(1.0000000000000002+0i) (0+0i)]

Quantum result (H->H, 1000 shots):
map[0:1000]

If superposition were a coin flip, we'd expect ~500 '0' and ~500 '1'.
Instead we get 1000 '0' and 0 '1' — proof of quantum interference!


q0 
 
 H 
 H 
 M

---

## Exercises

Try these on your own before peeking at the hints.

### Exercise 1 — Deterministic |1>

Build a circuit that **always** measures `1`. Run it for 1000 shots and display the histogram.

*Hint: Which single gate flips $|0\rangle$ to $|1\rangle$?*

In [8]:
%%
// Exercise 1: Build a circuit that always measures |1>
// Expected: all 1000 shots should give "1"
//
// Replace the ... with the correct gate call
// c, _ := builder.New("always-one", 1). ... .MeasureAll().Build()
// sim := statevector.New(1)
// counts, _ := sim.Run(c, 1000)
// gonbui.DisplayHTML(viz.Histogram(counts, viz.WithTitle("Exercise 1: Always |1>")))
fmt.Println("Uncomment the code above and fill in the gate!")

Uncomment the code above and fill in the gate!


### Exercise 2 — A 75/25 split

Create a circuit that measures `0` with 75% probability and `1` with 25% probability. Run it for 1000 shots and display the histogram.

*Hint: The RY gate rotates the qubit on the Bloch sphere. If you want $P(|0\rangle) = \cos^2(\theta/2) = 0.75$, solve for $\theta$. The `math.Acos` and `math.Sqrt` functions will help.*

In [9]:
%%
// Exercise 2: Build a circuit that gives 75% |0> and 25% |1>
// Expected: approximately 750 "0" and 250 "1" (±50)
//
// theta := 2 * math.Acos(math.Sqrt( ... ))
// c, _ := builder.New("75-25", 1).RY(theta, 0).MeasureAll().Build()
// sim := statevector.New(1)
// counts, _ := sim.Run(c, 1000)
// gonbui.DisplayHTML(viz.Histogram(counts, viz.WithTitle("Exercise 2: 75/25 split")))
_ = math.Sqrt // suppress unused import
fmt.Println("Uncomment the code above and fill in the probability!")

Uncomment the code above and fill in the probability!


---

## Key takeaways

1. **A qubit** is a unit vector in a 2D complex vector space, written $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$.

2. **Superposition** means $\alpha$ and $\beta$ can both be non-zero simultaneously. The qubit is not "secretly" 0 or 1 — it is genuinely in both states at once.

3. **The Born rule** says measurement outcome probabilities are the squared magnitudes of the amplitudes: $P(0) = |\alpha|^2$, $P(1) = |\beta|^2$.

4. **Interference** is the hallmark of quantum mechanics. Amplitudes can add constructively or destructively, as we saw with $H \cdot H|0\rangle = |0\rangle$. No classical probability model can reproduce this.

5. **The Bloch sphere** is a geometric way to visualize single-qubit states, with $|0\rangle$ at the north pole, $|1\rangle$ at the south pole, and superposition states on the surface.

---

**Next up:** Notebook 02 — Single-Qubit Gates, where we'll explore all the ways to manipulate a single qubit.